In [10]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (Conv1D, LSTM, Dense, Flatten,
                                     Bidirectional, Input, Conv1DTranspose)
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.model_selection import train_test_split
import joblib

df = pd.read_parquet('base_anomalias_cluster.parquet')
df.drop('K', axis=1, inplace=True)

# Visualizar las primeras filas
print(df.head())

#X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)


def procesar_cluster(df, cluster_id, n_steps=10):
      # filtramos los datos para un cluster y que no son anomalias, solo vamos a entrenar la red con datos normales
      df_filtrado = df[(df['cluster'] == cluster_id) & (df['es_anomalia_IQR_local'] == 0)].copy()

      df_filtrado = df_filtrado.sort_values('time')
      df_filtrado.reset_index(drop=True, inplace=True)
      df_filtrado = df_filtrado.set_index('time')

      # Agrupar por hora y promediar
      df_filtrado = df_filtrado.resample('h')[['presion', 'volumen', 'temperatura']].mean().dropna().reset_index()

      # --- Descomposición y uso de residuales ---
      # R++ dejamos de usar residuales al requererir un bloque mayor de datos al predecir
      # para aplicar seasonal_decompose con period=168, necesitariamos al menos ~250 datos (idealmente más
      for var in ['presion', 'volumen', 'temperatura']:
          try:
              resultado = seasonal_decompose(df_filtrado[var], model='additive', period=168, extrapolate_trend='freq')
              df_filtrado[f'{var}_resid'] = resultado.resid
          except:
              print(f"Fallo descomposición en {var} del cluster {cluster_id}")
              continue

      # Usamos solo residuales
      vars_resid = ['presion_resid', 'volumen_resid', 'temperatura_resid']
      df_clean = df_filtrado.dropna(subset=vars_resid)

      # Agregar características cíclicas de tiempo
      df_clean['hora'] = df_filtrado['time'].dt.hour
      df_clean['dia'] = df_filtrado['time'].dt.dayofweek

      df_clean['hora_sin'] = np.sin(2 * np.pi * df_clean['hora'] / 24)
      df_clean['hora_cos'] = np.cos(2 * np.pi * df_clean['hora'] / 24)
      df_clean['dia_sin'] = np.sin(2 * np.pi * df_clean['dia'] / 7)
      df_clean['dia_cos'] = np.cos(2 * np.pi * df_clean['dia'] / 7)

      # Variables de entrada: todas las que usas como features (7 en total)
      vars_resid = ['presion_resid', 'volumen_resid', 'temperatura_resid', 'hora_sin', 'hora_cos', 'dia_sin', 'dia_cos']
      # Variables objetivo
      target_features = ['presion_resid', 'volumen_resid', 'temperatura_resid']

      df_clean = df_clean[vars_resid]
      print(df_clean.head())

      # Escalado
      scaler = MinMaxScaler()
      data_scaled = scaler.fit_transform(df_clean[vars_resid])


      # Crear secuencias
      X, y = [], []
      for i in range(len(data_scaled) - n_steps):
          X.append(data_scaled[i:i+n_steps])
          y.append(data_scaled[i+n_steps])

      X, y = np.array(X), np.array(y)
      if len(X) < 10:
          return  # Saltar si no hay suficiente data

      # Construcción del modelo
      input_layer = Input(shape=(n_steps, 7))
      cnn = Conv1D(filters=64, kernel_size=3, dilation_rate=2, activation='relu', padding="same")(input_layer)
      cnn = Conv1D(filters=64, kernel_size=3, dilation_rate=4, activation='relu', padding="same")(cnn)
      lstm = Bidirectional(LSTM(50, return_sequences=False))(cnn)
      decoder = Dense(50, activation='relu')(lstm)
      decoder = Dense(3, activation='linear')(decoder)

      autoencoder = Model(inputs=input_layer, outputs=decoder)
      autoencoder.compile(optimizer='adam', loss='mse')
      # 5. Entrenamiento sin etiquetas
      autoencoder.fit(X, y, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

      # 6. Fine-tuning con Softmax (para clasificación de anomalías)
      classification_layer = Dense(2, activation='softmax')(lstm)  # Anomalía o normal
      classifier = Model(inputs=input_layer, outputs=classification_layer)
      classifier.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

      # Error y etiquetas
      predictions = autoencoder.predict(X)
      error = np.mean(np.square(predictions - y), axis=1)

      threshold = np.percentile(error, 95)
      labels = (error > threshold).astype(int)
      # Convertir a one-hot encoding para que tenga la forma (None, 2)
      labels = to_categorical(labels, num_classes=2)

      # 7. Fine-tuning con etiquetas
      classifier.fit(X, labels, epochs=20, batch_size=32)

      # 8. Detección de anomalías en nuevos datos
      pred_class = classifier.predict(X)
      df_clean = df_clean.iloc[n_steps:].reset_index(drop=True)
      df_clean['Anomalia'] = np.argmax(pred_class, axis=1)

      # Guardar modelos y scaler
      autoencoder.save(f"autoencoder_cluster_{cluster_id}.h5")
      classifier.save(f"classifier_cluster_{cluster_id}.h5")
      joblib.dump(scaler, f"scaler_cluster_{cluster_id}.pkl")

      # Resultado por cliente
      print(f"Cluster {cluster_id} - Anomalías: {df_clean['Anomalia'].sum()}")


   ID                time    presion  temperatura    volumen  mes  dia  hora  \
0   1 2019-01-31 06:00:00  17.729719    28.766576  26.102865    1    3     6   
1   1 2019-01-31 07:00:00  17.693210    28.138249  24.492739    1    3     7   
2   1 2019-01-31 09:00:00  17.717018    28.478988  26.036619    1    3     9   
3   1 2019-01-31 10:00:00  17.745201    28.377288  28.541832    1    3    10   
4   1 2019-01-31 11:00:00  17.662240    28.780606  26.991910    1    3    11   

           PV  cluster  es_anomalia_local_presion  es_anomalia_local_volumen  \
0  462.796468        2                      False                      False   
1  433.355181        2                      False                      False   
2  461.291240        2                      False                      False   
3  506.480551        2                      False                      False   
4  476.737584        2                      False                      False   

   es_anomalia_local_temperatura  es_a

In [11]:
procesar_cluster(df, cluster_id=0)
procesar_cluster(df, cluster_id=1)
procesar_cluster(df, cluster_id=2)
procesar_cluster(df, cluster_id=3)
procesar_cluster(df, cluster_id=4)
procesar_cluster(df, cluster_id=5)
procesar_cluster(df, cluster_id=6)
procesar_cluster(df, cluster_id=7)

   presion_resid  volumen_resid  temperatura_resid  hora_sin  hora_cos  \
0       0.026215      -1.725311          -0.093036  0.000000  1.000000   
1       0.046832      12.432073          -0.006569  0.258819  0.965926   
2       0.036246       4.002554           0.070246  0.500000  0.866025   
3       0.023705     -19.578102           0.041989  0.707107  0.707107   
4       0.080522     -15.654708          -0.128456  0.866025  0.500000   

   dia_sin  dia_cos  
0      0.0      1.0  
1      0.0      1.0  
2      0.0      1.0  
3      0.0      1.0  
4      0.0      1.0  
Epoch 1/50


ValueError: Dimensions must be equal, but are 7 and 3 for '{{node compile_loss/mse/sub}} = Sub[T=DT_FLOAT](data_1, functional_9_1/dense_16_1/BiasAdd)' with input shapes: [?,7], [?,3].

In [7]:
from tensorflow.keras.models import load_model
import joblib

def predecir_anomalia(df_secuencia, cluster_id):
    """
    df_secuencia: DataFrame con 10 filas y columnas: time, presion, volumen, temperatura
    cluster_id: ID del cluster correspondiente
    scaler_path: Ruta del MinMaxScaler entrenado
    autoencoder_path: Ruta del modelo autoencoder entrenado
    classifier_path: Ruta del modelo clasificador entrenado
    """

    scaler_path = f"autoencoder_cluster_{cluster_id}.h5"
    classifier_path = f"classifier_cluster_{cluster_id}.h5"
    scaler_path = f"scaler_cluster_{cluster_id}.pkl"

    df_secuencia = df_secuencia.sort_values('time').reset_index(drop=True)

    # Descomposición simple: usar diferencia respecto al promedio para residuales
    for var in ['presion', 'volumen', 'temperatura']:
        df_secuencia[f'{var}_resid'] = df_secuencia[var] - df_secuencia[var].mean()

    # Codificación de hora y día
    df_secuencia['hora'] = df_secuencia['time'].dt.hour
    df_secuencia['dia'] = df_secuencia['time'].dt.dayofweek

    df_secuencia['hora_sin'] = np.sin(2 * np.pi * df_secuencia['hora'] / 24)
    df_secuencia['hora_cos'] = np.cos(2 * np.pi * df_secuencia['hora'] / 24)
    df_secuencia['dia_sin'] = np.sin(2 * np.pi * df_secuencia['dia'] / 7)
    df_secuencia['dia_cos'] = np.cos(2 * np.pi * df_secuencia['dia'] / 7)

    vars_resid = ['presion_resid', 'volumen_resid', 'temperatura_resid',
                  'hora_sin', 'hora_cos', 'dia_sin', 'dia_cos']

    # Escalar
    scaler = joblib.load(scaler_path)
    data_scaled = scaler.transform(df_secuencia[vars_resid])

    X_input = np.expand_dims(data_scaled, axis=0)  # (1, n_steps, n_features)

    # Cargar modelos
    autoencoder = load_model(autoencoder_path)
    classifier = load_model(classifier_path)

    # Predicción
    pred_class = classifier.predict(X_input)
    clase = np.argmax(pred_class, axis=1)[0]

    return clase  # 1 = anómalo, 0 = normal

In [ ]:

predecir_anomalia(0)
